# 🤖 Model Training & Evaluation

Train Logistic Regression, Random Forest, XGBoost, and LightGBM.
Compare performance using ROC-AUC, F1, Precision, Recall.
Generate SHAP explainability plots.

In [ ]:
import sys, os
sys.path.insert(0, "../src")
from data_loader import load_config, load_raw_data, clean_data
from features import prepare_features
from model import build_models, evaluate_model, cross_validate_model, plot_results, plot_shap, save_model, print_summary
import pandas as pd
config = load_config("../config.yaml")
config["data"]["raw_path"] = "../" + config["data"]["raw_path"]
config["model"]["model_path"] = "../" + config["model"]["model_path"]
df = load_raw_data(config["data"]["raw_path"])
df_clean = clean_data(df, config)
X_train, X_test, y_train, y_test, feature_names = prepare_features(df_clean, config)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
models = build_models(config)
results = []
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    cv = cross_validate_model(model, X_train, y_train, config)
    res = evaluate_model(model, X_test, y_test, name)
    res["cv_auc"] = round(cv, 4)
    results.append(res)
print_summary(results)

In [ ]:
# Save best model
save_model(models["LightGBM"], config["model"]["model_path"])

# Plot results
plot_results(results, X_test, y_test, models, save_dir="../reports")

from IPython.display import Image
Image("../reports/roc_curves.png")

In [ ]:
# SHAP explainability
plot_shap(models["LightGBM"], X_test, feature_names, save_dir="../reports")
from IPython.display import Image
Image("../reports/shap_summary.png")